# **Search and A\* - Hands-On**

## **Task 0: Understand the Search Problem**

| Component | Specification |
|---|---|
| State S | Robot position (row, col) on a non-obstacle cell |
| Actions A | Up, Down, Left, Right: offsets (-1,0), (+1,0), (0,-1), (0,+1) |
| Transition T | (r,c) -> (r+dr, c+dc) if that cell is inside the map and not '#' |
| Initial state s0 | The S cell = (1,1) |
| Goal G | {(7,15)} |
| Cost c | 1 per move |

(a) The (row, col) position

(b) Moving outside the map or into a '#' cell.

(c) Yes. Each state and action gives exactly one next state; there is no randomness.

(d) A sequence of valid moves from (1,1) to (7,15). The optimal one has the fewest moves.

## **Task 1: Plan the Agent**

1. State: (row, col).
2. Warehouse: list of lists of characters parsed from the ASCII map.
3. Valid actions: try the 4 offsets; keep those inside the map and not '#'.
4. Goal test: state == goal, checked when the node is popped from the frontier.
5. Frontier: a min-heap of (f, tie_breaker, state), plus dictionaries g[state] and parent[state].
6. Path reconstruction: follow parent pointers from the goal back to the start, then reverse.

Reported at the end: solution found?, path, path length, number of states expanded.


## **Task 2: Ask an LLM to Generate A\***

**Prompt:**

I am implementing a simple goal-based search agent in Python.
The environment is a grid represented by an ASCII map. The agent starts at S
and must reach G. The symbols # represent obstacles and . represents free cells.
The agent can move up, down, left, or right, and every movement has cost 1.
Implement A* search.
Use Manhattan distance as the heuristic:
h(n) = |x − xG| + |y − yG|.
The program should:
- represent grid positions as states;
- maintain an appropriate frontier;
- calculate g(n), h(n) and f(n);
- avoid repeatedly expanding the same state;
- reconstruct the path when the goal is reached;
- report the path and its length;
- report the number of states expanded.
Keep the implementation simple and explain the main components of the code



## **Task 3: Test the Generated Program**

In [1]:
import heapq, math
from collections import deque

WAREHOUSE = """\
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################"""

ACTIONS = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}

def parse(text):
    grid = [list(r) for r in text.strip().splitlines()]
    start = goal = None
    for r, row in enumerate(grid):
        for c, ch in enumerate(row):
            if ch == "S": start = (r, c)
            elif ch == "G": goal = (r, c)
    return grid, start, goal

def successors(grid, state):
    r, c = state
    for name, (dr, dc) in ACTIONS.items():
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[nr]) and grid[nr][nc] != "#":
            yield name, (nr, nc), 1

def manhattan(s, g): return abs(s[0]-g[0]) + abs(s[1]-g[1])
def euclid(s, g):    return math.hypot(s[0]-g[0], s[1]-g[1])
def zero(s, g):      return 0

def reconstruct(parent, state):
    path = []
    while state is not None:
        path.append(state)
        state = parent[state]
    return path[::-1]

def astar(text, h=manhattan):
    grid, start, goal = parse(text)
    counter = 0
    frontier = [(h(start, goal), counter, start)]   # (f, tie, state)
    g = {start: 0}
    parent = {start: None}
    closed = set()
    expanded = 0
    while frontier:
        f, _, s = heapq.heappop(frontier)
        if s in closed:
            continue
        if s == goal:
            path = reconstruct(parent, s)
            return dict(found=True, path=path, length=len(path)-1, expanded=expanded)
        closed.add(s)
        expanded += 1
        for _, s2, cost in successors(grid, s):
            g2 = g[s] + cost
            if s2 not in closed and g2 < g.get(s2, float("inf")):
                g[s2] = g2
                parent[s2] = s
                counter += 1
                heapq.heappush(frontier, (g2 + h(s2, goal), counter, s2))
    return dict(found=False, path=None, length=None, expanded=expanded)

def draw(text, path):
    grid, _, _ = parse(text)
    for (r, c) in (path or [])[1:-1]:
        grid[r][c] = "*"
    return "\n".join("".join(r) for r in grid)

def report(name, res, text=None):
    print(f"--- {name} ---")
    print(f"solution found: {res['found']} | path length: {res['length']} | states expanded: {res['expanded']}")
    if res["found"] and text:
        print(draw(text, res["path"]))
        print("path:", res["path"])

In [2]:
report("Test 1: warehouse", astar(WAREHOUSE), WAREHOUSE)

--- Test 1: warehouse ---
solution found: True | path length: 40 | states expanded: 63
#################
#S****#*********#
#.###*#*#######*#
#...#*#*******#*#
###.#*#######*#*#
#...#*********#*#
#.###########.#*#
#.............#G#
#################
path: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]


**Test 1: Original Warehouse**

Result: a path was found, length 40, 63 states expanded.

Path: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]

In [3]:
T2 = "#####\n#SG##\n#####"
report("Test 2: trivial", astar(T2), T2)

--- Test 2: trivial ---
solution found: True | path length: 1 | states expanded: 1
#####
#SG##
#####
path: [(1, 1), (1, 2)]


**Test 2: Trivial Case**

Result: found, length 1, 1 state expanded.

In [4]:
T3 = "#######\n#S....#\n###.###\n#...#G#\n#######"
report("Test 3: no solution", astar(T3))

--- Test 3: no solution ---
solution found: False | path length: None | states expanded: 9


**Test 3: No Solution**

Result: found = False, 9 states expanded (all cells reachable from S). The program stops instead of looping, because the closed set prevents revisiting states.

In [5]:
T4 = ("#######\n"
      "#S...G#\n"
      "#.###.#\n"
      "#.....#\n"
      "#######")
report("Test 4: alternative paths", astar(T4), T4)

--- Test 4: alternative paths ---
solution found: True | path length: 4 | states expanded: 4
#######
#S***G#
#.###.#
#.....#
#######
path: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5)]


**Test 4: Alternative Paths**

Result: two routes exist (top row = 4 moves, bottom loop = 8 moves). A* returned the 4-move route, so it is a shortest path. BFS (Task 5) also gives length 4.

## **Task 4: Inspect the A\* Algorithm**

| Concept | Where does it Appear in the Code|
|---|---|
| State | (row, col) tuples; start comes from parse() |
| Action | the ACTIONS dictionary |
| Transition | nr, nc = r+dr, c+dc inside successors() |
| Goal test | if s == goal: inside astar() |
| g(n) | the g dictionary, g2 = g[s] + cost |
| h(n) | the manhattan() function, called as h(s2, goal) |
| f(n) | g2 + h(s2, goal), the first item in each heap entry |
| Frontier | the frontier list used as a heapq min-heap |
| Visited states | the closed set |
| Path reconstruction | reconstruct(parent, state) |

(a) A priority queue from heapq.

(b) heappop takes the entry with the smallest f; ties go to the one inserted first.

(c) In manhattan(); it is called when a successor is pushed and once for the start.

(d) Yes, implicitly: f = g2 + h(s2, goal) is computed when pushing and used as the heap key.

(e) The closed set: popped states already closed are skipped and closed neighbours are not pushed. The check g2 < g.get(s2, inf) also avoids queueing worse routes to the same cell.

## **Task 5: Compare A\* with Blind Search**



In [6]:
def bfs(text):
    grid, start, goal = parse(text)
    frontier = deque([start])
    parent = {start: None}
    expanded = 0
    while frontier:
        s = frontier.popleft()
        if s == goal:
            path = reconstruct(parent, s)
            return dict(found=True, path=path, length=len(path)-1, expanded=expanded)
        expanded += 1
        for _, s2, _ in successors(grid, s):
            if s2 not in parent:
                parent[s2] = s
                frontier.append(s2)
    return dict(found=False, path=None, length=None, expanded=expanded)

report("BFS", bfs(WAREHOUSE), WAREHOUSE)
report("A*",  astar(WAREHOUSE), WAREHOUSE)

--- BFS ---
solution found: True | path length: 40 | states expanded: 63
#################
#S****#*********#
#.###*#*#######*#
#...#*#*******#*#
###.#*#######*#*#
#...#*********#*#
#.###########.#*#
#.............#G#
#################
path: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]
--- A* ---
solution found: True | path length: 40 | states expanded: 63
#################
#S****#*********#
#.###*#*#######*#
#...#*#*******#*#
###.#*#######*#*#
#...#*********#*#
#.###########.#*#
#.............#G#
#################
path: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (

| Measure | BFS | A* |
|---|---|---|
| Solution found | Yes | Yes |
| Path length | 40 | 40 |
| States expanded | 63 | 63 |

(a) Yes, both found a solution.

(b) Yes, both found paths of length 40.

(c) Neither; both expanded 63 states.

(d) A* can expand fewer states because the heuristic favours cells that look closer to the goal, while BFS expands level by level. Here it did not help: the maze winds away from the goal, so the shortest path (40) is double the start's Manhattan estimate (20). Nearly every cell therefore has g + h < 40, and A* cannot skip any. Manhattan distance ignores walls, so it is a weak heuristic in a maze.

## **Task 6: Investigate the Heuristic**

Manhattan distance is appropriate because with 4-direction moves of cost 1, any path needs at least |dRow| + |dCol| moves, and walls can only add more. So h(n) <= h*(n): it is admissible (and consistent).

In [7]:
def run_with(name, h):
    r = astar(WAREHOUSE, h)
    print(f"{name:14s} found={r['found']}  length={r['length']}  expanded={r['expanded']}")

run_with("Manhattan",     manhattan)
run_with("h = 0",         zero)
run_with("Euclidean",     euclid)
run_with("2 x Manhattan", lambda s, g: 2*manhattan(s, g))
run_with("5 x Manhattan", lambda s, g: 5*manhattan(s, g))

Manhattan      found=True  length=40  expanded=63
h = 0          found=True  length=40  expanded=63
Euclidean      found=True  length=40  expanded=63
2 x Manhattan  found=True  length=40  expanded=63
5 x Manhattan  found=True  length=48  expanded=63


- h = 0: A* becomes uniform-cost search (like BFS). Still optimal, but uses no goal information.
- Euclidean: admissible but less informed than Manhattan, so it cannot prune more.
- 2 x Manhattan: not admissible in general. It happened to stay optimal here, which does not prove it is safe.
- 5 x Manhattan: too optimistic. A* returned a 48-step path instead of the optimal 40.

When the heuristic overestimates the true cost, A* can return a suboptimal path, because it stops following the cheaper route. A more aggressive heuristic makes A* behave more like greedy search: it may expand fewer nodes, but the solution is no longer guaranteed optimal.

## **Task 7: Evaluate the LLM-Generated Agent**

1. Correct immediately: parsing, successor generation, the heap frontier, the closed set and path reconstruction.
2. Bugs or design problems: none in the final logic. Things to watch for: the goal test must happen when a node is popped (not when pushed) to guarantee optimality; the heap needs a tie-breaker so states are never compared; BFS and A* must count "expanded" the same way.
3. How found: by comparing BFS and A* outputs and expansion counts on the same map.
4. Unfamiliar terms: heapq, stale heap entries, tie-breaking counter.
5. Modified: Nothing.
6. Most useful tests: Test 3 (no solution, checks termination) and Test 4 (alternative paths, checks optimality against BFS).
7. Trust without testing? No. A plausible path on one map does not show the algorithm is correct.
8. Learned about A*: its benefit depends on how informative the heuristic is (here Manhattan gave no saving over BFS), and an over-optimistic heuristic loses optimality.

Designed by me: the problem formulation, the agent design (Task 1), and the tests.

Suggested by the LLM: the code structure and the heapq/closed-set approach.

Accepted: Entire code.

Changed: Nothing.

Tested: Tests 1-4, BFS comparison, heuristic variants.

## **Final Reflection**

1. Formulating the problem first forces decisions about what a state is, which moves are legal and what counts as a solution. Without that there is nothing to check the code against. It also made the tests easy to design: a trivial case, an unreachable goal and a multi-path map all follow from the formulation.

2. A* is informed because it uses problem-specific knowledge, the heuristic h(n), to estimate the remaining cost. It orders the frontier by g + h, combining the cost so far with an estimate of the cost to come, whereas BFS and DFS use only the graph structure.

3. The heuristic controls both correctness and efficiency. An admissible heuristic guarantees optimal paths, and a more accurate one expands fewer nodes. In our experiments h = 0 gave no guidance, Manhattan helped little because of the walls, and 5 x Manhattan returned a 48-step path instead of the optimal 40.

4. The LLM turned the design into working code quickly, supplied standard idioms (heap, closed set, parent pointers) and helped generate test maps. It did not decide what needed testing and did not verify its own output.

5. The code could loop forever on an unreachable goal, return a non-shortest path (for example by testing the goal when pushing) or miscount expansions so that algorithm comparisons are meaningless. Plausible output is not validated behaviour.